## 1. Brevitas Import, FINN Export 
En primer lugar importamos el modelo generado con Brevitas y lo exportamos a un modelo de onnx finn.

In [9]:
import qonnx.core.modelwrapper as mw
print("✅ QONNX cargado")
from qonnx.core.modelwrapper import ModelWrapper
print("✅ FINN Core cargado")

✅ QONNX cargado
✅ FINN Core cargado


In [10]:
import torch
print(f"✅ Torch cargado. Versión: {torch.__version__}")
import brevitas
print("✅ Brevitas cargado")

✅ Torch cargado. Versión: 1.7.1
✅ Brevitas cargado


In [11]:
import os
# Forzamos a Torch a ignorar la GPU y usar solo CPU
os.environ["CUDA_VISIBLE_DEVICES"] = ""
import torch
print("✅ Torch cargado en modo CPU")

✅ Torch cargado en modo CPU


In [12]:
import gc
import os
import sys

def cleanup_safe():
    # Solo limpieza de Python estándar
    gc.collect()
    # Imprime la ruta para ver si hay algún "polizón" en el PATH
    print("Python Path actual:", sys.path[0])

cleanup_safe()

Python Path actual: /home/miguel_tfg/FINN-v0.9-Stable/src


In [15]:
import json
import numpy as np
import os
import shutil
import warnings
from copy import deepcopy
from distutils.dir_util import copy_tree
from finn.util.basic import make_build_dir
from finn.util.visualization import showInNetron
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg
import onnx
from finn.util.test import get_test_model_trained
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.general import (
    ApplyConfig,
    GiveReadableTensorNames,
    GiveUniqueNodeNames,
    RemoveStaticGraphInputs,
    RemoveUnusedTensors,
)
import brevitas.onnx as bo
import brevitas.nn as qnn
from brevitas.quant import Int8Bias
from brevitas.core.restrict_val import RestrictValueType
import torch 
import torch.nn as nn
import torch.nn.functional as F
from qonnx.util.cleanup import cleanup_model
from qonnx.util.config import extract_model_config_to_json
from shutil import copy

import finn.transformation.fpgadataflow.convert_to_hls_layers as to_hls
import finn.transformation.streamline.absorb as absorb
from finn.analysis.fpgadataflow.dataflow_performance import dataflow_performance
from finn.analysis.fpgadataflow.exp_cycles_per_layer import exp_cycles_per_layer
from finn.analysis.fpgadataflow.hls_synth_res_estimation import hls_synth_res_estimation
from finn.analysis.fpgadataflow.op_and_param_counts import (
    aggregate_dict_keys,
    op_and_param_counts,
)
from finn.analysis.fpgadataflow.res_estimation import (
    res_estimation,
    res_estimation_complete,
)
from finn.builder.build_dataflow_config import (
    DataflowBuildConfig,
    DataflowOutputType,
    ShellFlowType,
    VerificationStepType,
)
from finn.core.onnx_exec import execute_onnx
from finn.core.rtlsim_exec import rtlsim_exec
from finn.core.throughput_test import throughput_test_rtlsim
from finn.transformation.fpgadataflow.annotate_cycles import AnnotateCycles
from finn.transformation.fpgadataflow.compile_cppsim import CompileCppSim
from finn.transformation.fpgadataflow.create_dataflow_partition import (
    CreateDataflowPartition,
)
from finn.transformation.fpgadataflow.create_stitched_ip import CreateStitchedIP
from finn.transformation.fpgadataflow.derive_characteristic import (
    DeriveCharacteristic,
    DeriveFIFOSizes,
)
from finn.transformation.fpgadataflow.insert_tlastmarker import InsertTLastMarker
from finn.transformation.fpgadataflow.hlssynth_ip import HLSSynthIP
from finn.transformation.fpgadataflow.insert_dwc import InsertDWC
from finn.transformation.fpgadataflow.insert_fifo import InsertFIFO
from finn.transformation.fpgadataflow.make_pynq_driver import MakePYNQDriver
from finn.transformation.fpgadataflow.make_zynq_proj import ZynqBuild
from finn.transformation.fpgadataflow.prepare_cppsim import PrepareCppSim
from finn.transformation.fpgadataflow.prepare_ip import PrepareIP
from finn.transformation.fpgadataflow.prepare_rtlsim import PrepareRTLSim
from finn.transformation.fpgadataflow.replace_verilog_relpaths import (
    ReplaceVerilogRelPaths,
)
from finn.transformation.fpgadataflow.set_exec_mode import SetExecMode
from finn.transformation.fpgadataflow.set_fifo_depths import (
    InsertAndSetFIFODepths,
    RemoveShallowFIFOs,
    SplitLargeFIFOs,
)
from finn.transformation.fpgadataflow.set_folding import SetFolding
from finn.transformation.fpgadataflow.synth_ooc import SynthOutOfContext
from finn.transformation.fpgadataflow.vitis_build import VitisBuild
from finn.transformation.move_reshape import RemoveCNVtoFCFlatten
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from finn.transformation.qonnx.quant_act_to_multithreshold import (
    default_filter_function_generator,
)
from finn.transformation.streamline import Streamline
from finn.transformation.streamline.reorder import MakeMaxPoolNHWC
from finn.util.basic import (
    get_rtlsim_trace_depth,
    pyverilate_get_liveness_threshold_cycles,
)
from finn.util.pyverilator import verilator_fifosim
from finn.util.test import execute_parent


In [16]:
state_dict = torch.load("state_dict_quantized_last.pth", map_location="cpu")

for k, v in state_dict.items():
    try:
        print(k, v.shape)
    except:
        print(k, v)

quant_input.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value torch.Size([])
fc1.weight torch.Size([256, 576])
fc1.bias torch.Size([256])
relu1.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value torch.Size([])
fc2.weight torch.Size([128, 256])
fc2.bias torch.Size([128])
relu2.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value torch.Size([])
fx.weight torch.Size([16, 128])
fx.bias torch.Size([16])
relu3.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value torch.Size([])
predictor.weight torch.Size([2, 16])
predictor.bias torch.Size([2])
quant_output.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value torch.Size([])


In [14]:

class BobEsponja(nn.Module):
    def __init__(self, classes, b_witdh, in_width):
        super(BobEsponja, self).__init__()
        self.QuantInput = qnn.QuantIdentity(return_quant_tensor=True, bit_width=in_width)
        self.conv1 = qnn.QuantConv2d(in_channels=1, out_channels=16, kernel_size=(3, 3), weight_bit_width = b_witdh, return_quant_tensor=True, bias = False, bias_quant=Int8Bias, padding = 1)
        self.relu1 = qnn.QuantReLU(bit_width=b_witdh, return_quant_tensor=True)
        self.maxpool1 = qnn.QuantMaxPool2d(kernel_size=(2, 2), stride=(2, 2))
        self.conv2 = qnn.QuantConv2d(in_channels=16, out_channels=32, kernel_size=(3, 3), weight_bit_width = b_witdh, return_quant_tensor=True, bias = True, bias_quant=Int8Bias, padding = 1)
        self.relu2 = qnn.QuantReLU(bit_width=b_witdh, return_quant_tensor=True)
        self.maxpool2 = qnn.QuantMaxPool2d(kernel_size=(2, 2), stride=(2, 2))
        self.conv3 = qnn.QuantConv2d(in_channels=32, out_channels=64, kernel_size=(3, 3), weight_bit_width = b_witdh, return_quant_tensor=True, bias = True, bias_quant=Int8Bias, padding = 1)
        self.relu3 = qnn.QuantReLU(bit_width=b_witdh, return_quant_tensor=True)
        self.maxpool3 = qnn.QuantMaxPool2d(kernel_size=(2, 2), stride=(2, 2))

        self.fc1 = qnn.QuantLinear(in_features=4096, out_features=128, weight_bit_width=4, return_quant_tensor=True, bias = True, bias_quant=Int8Bias)
        self.relu4 = qnn.QuantReLU(bit_width=b_witdh, return_quant_tensor=True)
        self.fc2 = qnn.QuantLinear(in_features=128, out_features=classes, return_quant_tensor=True, weight_bit_width=b_witdh, bias = False)

    def forward(self, x):
        x = self.QuantInput(x)
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.maxpool1(x)
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.maxpool2(x)
        x = self.conv3(x)
        x = self.relu3(x)
        x = self.maxpool3(x)
        x = x.reshape(x.size(0),-1)
        x = self.fc1(x) 
        x = self.relu4(x)
        out = self.fc2(x)
        return out 

model = BobEsponja(14,4,8)
model.load_state_dict(torch.load("state_dict_quantized_last.pth"))
model_file_name = "model.onnx"
bo.export_finn_onnx(model, (1,1,64,64), model_file_name)

# showInNetron("model.onnx")


RuntimeError: Error(s) in loading state_dict for BobEsponja:
	Missing key(s) in state_dict: "QuantInput.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value", "conv1.weight", "conv2.weight", "conv2.bias", "conv3.weight", "conv3.bias", "relu4.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value". 
	Unexpected key(s) in state_dict: "quant_input.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value", "fx.weight", "fx.bias", "predictor.weight", "predictor.bias", "quant_output.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value", "fc2.bias". 
	size mismatch for fc1.weight: copying a param with shape torch.Size([256, 576]) from checkpoint, the shape in current model is torch.Size([128, 4096]).
	size mismatch for fc1.bias: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for fc2.weight: copying a param with shape torch.Size([128, 256]) from check

In [7]:
def verify_step(
    model: ModelWrapper,
    cfg: DataflowBuildConfig,
    step_name: str,
    need_parent: bool,
    rtlsim_pre_hook=None,
):
    print("Running verification for " + step_name)
    verify_out_dir = cfg.output_dir + "/verification_output"
    intermediate_models_dir = cfg.output_dir + "/intermediate_models"
    os.makedirs(verify_out_dir, exist_ok=True)
    (in_npy_all, exp_out_npy_all) = cfg._resolve_verification_io_pair()
    bsize_in = in_npy_all.shape[0]
    bsize_out = exp_out_npy_all.shape[0]
    assert bsize_in == bsize_out, "Batch sizes don't match for verification IO pair"
    all_res = True
    for b in range(bsize_in):
        in_npy = np.expand_dims(in_npy_all[b], axis=0)
        exp_out_npy = np.expand_dims(exp_out_npy_all[b], axis=0)
        if need_parent:
            assert (
                cfg.save_intermediate_models
            ), "Enable save_intermediate_models for verification"
            parent_model_fn = intermediate_models_dir + "/dataflow_parent.onnx"
            child_model_fn = intermediate_models_dir + "/verify_%s.onnx" % step_name
            model.save(child_model_fn)
            parent_model = ModelWrapper(parent_model_fn)
            out_tensor_name = parent_model.graph.output[0].name
            exp_ishape = parent_model.get_tensor_shape(parent_model.graph.input[0].name)
            if in_npy.shape != exp_ishape:
                print(
                    "Verification input has shape %s while model expects %s"
                    % (str(in_npy.shape), str(exp_ishape))
                )
                print("Attempting to force model shape on verification input")
                in_npy = in_npy.reshape(exp_ishape)
            out_dict = execute_parent(
                parent_model_fn, child_model_fn, in_npy, return_full_ctx=True
            )
            out_npy = out_dict[out_tensor_name]
        else:
            inp_tensor_name = model.graph.input[0].name
            out_tensor_name = model.graph.output[0].name
            exp_ishape = model.get_tensor_shape(inp_tensor_name)
            if in_npy.shape != exp_ishape:
                print(
                    "Verification input has shape %s while model expects %s"
                    % (str(in_npy.shape), str(exp_ishape))
                )
                print("Attempting to force model shape on verification input")
                in_npy = in_npy.reshape(exp_ishape)
            inp_dict = {inp_tensor_name: in_npy}
            if rtlsim_pre_hook is not None:
                out_dict = rtlsim_exec(model, inp_dict, pre_hook=rtlsim_pre_hook)
            else:
                out_dict = execute_onnx(model, inp_dict, True)
            out_npy = out_dict[out_tensor_name]
        exp_oshape = exp_out_npy.shape
        if out_npy.shape != exp_oshape:
            print(
                "Verification output has shape %s while model produces %s"
                % (str(exp_oshape), str(out_npy.shape))
            )
            print("Attempting to force model shape on verification output")
            out_npy = out_npy.reshape(exp_oshape)

        res = np.isclose(exp_out_npy, out_npy, atol=1e-3).all()
        all_res = all_res and res
        res_to_str = {True: "SUCCESS", False: "FAIL"}
        res_str = res_to_str[res]
        if cfg.verify_save_full_context:
            verification_output_fn = verify_out_dir + "/verify_%s_%d_%s.npz" % (
                step_name,
                b,
                res_str,
            )
            np.savez(verification_output_fn, **out_dict)
        else:
            verification_output_fn = verify_out_dir + "/verify_%s_%d_%s.npy" % (
                step_name,
                b,
                res_str,
            )
            np.save(verification_output_fn, out_npy)
        if cfg.verify_save_rtlsim_waveforms:
            vcd_path = model.get_metadata_prop("rtlsim_trace")
            if vcd_path is not None and os.path.isfile(vcd_path):
                new_vcd_path = vcd_path.replace(".vcd", "_%d.vcd" % b)
                shutil.move(vcd_path, new_vcd_path)
    print("Verification for %s : %s" % (step_name, res_to_str[all_res]))
    
def prepare_for_stitched_ip_rtlsim(verify_model, cfg):
    need_restitch = False
    # rtlsim only supports certain impl_style for some nodes
    # StreamingFIFO must have impl_style=rtl
    for fifo_layer in verify_model.get_nodes_by_op_type("StreamingFIFO"):
        inst = getCustomOp(fifo_layer)
        if inst.get_nodeattr("impl_style") != "rtl":
            inst.set_nodeattr("impl_style", "rtl")
            inst.set_nodeattr("code_gen_dir_ipgen", "")
            inst.set_nodeattr("ipgen_path", "")
            need_restitch = True
    # StreamingDataWidthConverter must have impl_style=hls
    for dwc_layer in verify_model.get_nodes_by_op_type(
        "StreamingDataWidthConverter_Batch"
    ):
        inst = getCustomOp(dwc_layer)
        if inst.get_nodeattr("impl_style") != "hls":
            inst.set_nodeattr("impl_style", "hls")
            inst.set_nodeattr("code_gen_dir_ipgen", "")
            inst.set_nodeattr("ipgen_path", "")
            need_restitch = True
    # if we've made alterations to the model, need to do some re-prep
    if need_restitch:
        print("Need to regen/re-stitch some IP for STITCHED_IP_RTLSIM")
        verify_model = verify_model.transform(
            PrepareIP(cfg._resolve_fpga_part(), cfg._resolve_hls_clk_period())
        )
        verify_model = verify_model.transform(HLSSynthIP())
        verify_model = verify_model.transform(
            CreateStitchedIP(
                cfg._resolve_fpga_part(),
                cfg.synth_clk_period_ns,
                vitis=False,
            )
        )
    # set top-level prop for stitched-ip rtlsim and launch
    verify_model.set_metadata_prop("exec_mode", "rtlsim")
    # TODO make configurable
    # verify_model.set_metadata_prop("rtlsim_trace", "trace.vcd")
    return verify_model


## 2. Build Dataflow
Una vez que el modelo ha sido exportado, procedemos a implementar uno por uno los diferentes steps presentes en build_dataflow, modificándolos para nuestra red.

### 0. Step_ONNX_to_FINN

In [8]:
final_output_dir = "output_final"

#Delete previous run results if exist
if os.path.exists(final_output_dir):
    shutil.rmtree(final_output_dir)
    print("Previous run results deleted!")

cfg = build.DataflowBuildConfig(
    output_dir          = final_output_dir,
    folding_config_file = "folding_config_3cnv.json",
    auto_fifo_depths    = True,
    mvau_wwidth_max     = 36,
    stitched_ip_gen_dcp = True,
    # target_fps          = 100,
    split_large_fifos   = True,
    synth_clk_period_ns = 10.0,
    board               = "Pynq-Z2",
    default_mem_mode    = "const",
    shell_flow_type     = build_cfg.ShellFlowType.VIVADO_ZYNQ,
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
        build_cfg.DataflowOutputType.STITCHED_IP,
        build_cfg.DataflowOutputType.RTLSIM_PERFORMANCE,
        build_cfg.DataflowOutputType.OOC_SYNTH,
    ]
)

Previous run results deleted!


### 1. Step_tidy_up
Run the tidy-up step on given model. This includes shape and datatype inference, constant folding, and giving nodes and tensors better names.

In [14]:
from qonnx.core.datatype import DataType
from qonnx.transformation.insert_topk import InsertTopK

model = ModelWrapper("model.onnx")
# # Tidy-up Transformations
# global_inp_name = model.graph.input[0].name
# model.set_tensor_datatype(global_inp_name, DataType["UINT8"])
# model = model.transform(InsertTopK(k=1))
# Garantiza que todos los tensores tengan un tamaño o forma especificados
model = model.transform(InferShapes())
# Sustituye la salida de nodos cuya entrada es una constante con resultados precalculados
model = model.transform(FoldConstants())
# Da nombres únicos a cada nodo del graph completo

model = model.transform(GiveUniqueNodeNames())
# Da nombres a los tensores internos del graph
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
# Elimina cualquier entrada del graph de nivel superior que tenga inicializadores.
model = model.transform(RemoveStaticGraphInputs())

if VerificationStepType.TIDY_UP_PYTHON in cfg._resolve_verification_steps():
    verify_step(model, cfg, "initial_python", need_parent=False)

model.save("model.onnx")
# showInNetron("model.onnx")

/home/miguel_tfg/FINN-v0.9-Stable/deps/qonnx/src/qonnx/core/modelwrapper.py:93: UserWarning: Some old-style domain attributes were automatically converted to new-style,
                i.e. domain=finn to domain=qonnx.custom_op.<general|fpgadataflow|...>
  warnings.warn(


In [15]:
from finn.util.pytorch import ToTensor
from qonnx.transformation.merge_onnx_models import MergeONNXModels

model = ModelWrapper("model.onnx")
global_inp_name = model.graph.input[0].name
ishape = model.get_tensor_shape(global_inp_name)
# preprocessing: torchvision's ToTensor divides uint8 inputs by 255

totensor_pyt = ToTensor()
chkpt_preproc_name = "model_preproc.onnx"
bo.export_finn_onnx(totensor_pyt, ishape, chkpt_preproc_name)

# join preprocessing and core model
pre_model = ModelWrapper(chkpt_preproc_name)
model = model.transform(MergeONNXModels(pre_model))
# add input quantization annotation: UINT8
global_inp_name = model.graph.input[0].name
model.set_tensor_datatype(global_inp_name, DataType["UINT8"])

# postprocessing: insert Top-1 node at the end

model = model.transform(InsertTopK(k=1))

model.save("model.onnx")
# showInNetron("model.onnx")

/home/miguel_tfg/FINN-v0.9-Stable/deps/qonnx/src/qonnx/transformation/infer_data_layouts.py:119: UserWarning: Assuming 4D input is NCHW
  warnings.warn("Assuming 4D input is NCHW")


In [16]:
model = ModelWrapper("model.onnx")
# # Tidy-up Transformations
# Garantiza que todos los tensores tengan un tamaño o forma especificados
model = model.transform(InferShapes())
# Sustituye la salida de nodos cuya entrada es una constante con resultados precalculados
model = model.transform(FoldConstants())
# Da nombres únicos a cada nodo del graph completo
model = model.transform(GiveUniqueNodeNames())
# Da nombres a los tensores internos del graph
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
# Elimina cualquier entrada del graph de nivel superior que tenga inicializadores.
model = model.transform(RemoveStaticGraphInputs())

if VerificationStepType.TIDY_UP_PYTHON in cfg._resolve_verification_steps():
    verify_step(model, cfg, "initial_python", need_parent=False)
    
model.save("model.onnx")
# showInNetron("model.onnx")

### 2. Step_Streamline
Run streamlining on given model. Streamlining involves moving floating pointscale/shift parameters around, collapsing adjacent ones into a single parameter,then absorbing the scale/shift into the following `MultiThreshold` node. Streamlining requires careful topology design and cannot be applied to all topologies.

In [17]:
from finn.transformation.streamline import Streamline
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
from qonnx.transformation.bipolar_to_xnor import ConvertBipolarMatMulToXnorPopcount
import finn.transformation.streamline.absorb as absorb
from finn.transformation.streamline.reorder import MakeMaxPoolNHWC, MoveScalarLinearPastInvariants
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.general import RemoveUnusedTensors

model = ModelWrapper("model.onnx")
# Mueve operaciones lineales escalares (mul, add) más allá de funciones que son invariantes a ellas. 
# En concreto, iguala y transforma los siguientes patrones: f(x*C) -> f(x) * C f(x+C) -> f(x) + C donde x 
# es una entrada dinámica, C es un tensor constante. 
# Las f conocidas que obedecen esta propiedad son: Reshape, Flatten, Transpose, GlobalAveragePool
model = model.transform(MoveScalarLinearPastInvariants())
# Absorbe la bias escalar originado por la exportación de int con signo de vuelta a MultiThreshold y 
# reevalúa el tipo de datos de salida.
model = model.transform(absorb.AbsorbSignBiasIntoMultiThreshold())
model = model.transform(Streamline())
# Tras aplicar las transformaciones anteriores principalmente me ha absorvido todas las operaciones
# de Add, Mul y Div en nodos MultiThreshold, excepto los que están a la salida (antes de TopK)

# Sustituye las capas Conv por pares de capas Im2Col-MatMul, más capas Transpose para mantener 
# la disposición original de los datos.
model = model.transform(LowerConvsToMatMul())
# Para (NCHWTranspose -> MultiThreshold) mueve Transpose después de MultiThreshold 
# y establece su data_layout en NHWC.
model = model.transform(absorb.AbsorbTransposeIntoMultiThreshold())
# Convierte (MaxPool, NHWCTranspose) en (NHWCTranspose, MaxPoolNHWC) 
# y (NCHWTranspose, MaxPool) en (MaxPoolNHWC, NCHWTranspose).
model = model.transform(MakeMaxPoolNHWC())
model = model.transform(absorb.AbsorbConsecutiveTransposes())
model = model.transform(Streamline())

model = model.transform(absorb.AbsorbScalarMulAddIntoTopK())
# Intenta inferir la información de las anotaciones de disposición de datos para todos los tensores 
# de entrada/intermedio/salida basándose en las entradas y el tipo de nodo.
model = model.transform(InferDataLayouts())
model = model.transform(RemoveUnusedTensors())

if VerificationStepType.STREAMLINED_PYTHON in cfg._resolve_verification_steps():
    verify_step(model, cfg, "streamlined_python", need_parent=False)
    
model.save("model.onnx")
# showInNetron("model.onnx")

### 3. Step_Convert_to_hls
Convert eligible nodes to `HLSCustomOp` subclasses that represent HLS layers. Which nodes and particular configurations can be converted to HLS is limited, see the source code of the `convert_to_hls` module for more.

In [18]:
import finn.transformation.fpgadataflow.convert_to_hls_layers as to_hls
from finn.transformation.fpgadataflow.create_dataflow_partition import (
    CreateDataflowPartition,
)
from finn.transformation.move_reshape import RemoveCNVtoFCFlatten
from qonnx.custom_op.registry import getCustomOp
from qonnx.transformation.infer_data_layouts import InferDataLayouts

model = ModelWrapper("model.onnx")
# choose the memory mode for the MVTU units, decoupled or const
mem_mode = "decoupled"

model = model.transform(to_hls.InferQuantizedMatrixVectorActivation(mem_mode))
# input quantization (if any) to standalone thresholding
model = model.transform(to_hls.InferThresholdingLayer())
model = model.transform(to_hls.InferConvInpGen())
model = model.transform(to_hls.InferStreamingMaxPool())
# TopK to LabelSelect
model = model.transform(to_hls.InferLabelSelectLayer())
# get rid of Reshape(-1, 1) operation between hlslib nodes
model = model.transform(RemoveCNVtoFCFlatten())
# get rid of Tranpose -> Tranpose identity seq
model = model.transform(absorb.AbsorbConsecutiveTransposes())
# infer tensor data layouts
model = model.transform(InferDataLayouts())

model.save("model.onnx")
# showInNetron("model.onnx")

### 4. Step_Create_dataflow_partition
Separate consecutive groups of HLSCustomOp nodes into StreamingDataflowPartition nodes, which point to a separate ONNX file. Dataflow accelerator synthesis can only be performed on those HLSCustomOp sub-graphs.

In [19]:
model = ModelWrapper("model.onnx")

parent_model = model.transform(
    CreateDataflowPartition(
        partition_model_dir=cfg.output_dir
        + "/intermediate_models/supported_op_partitions"
    )
)
sdp_nodes = parent_model.get_nodes_by_op_type("StreamingDataflowPartition")
assert len(sdp_nodes) == 1, "Only a single StreamingDataflowPartition supported."
sdp_node = sdp_nodes[0]
sdp_node = getCustomOp(sdp_node)
dataflow_model_filename = sdp_node.get_nodeattr("model")
if cfg.save_intermediate_models:
    parent_model.save(cfg.output_dir + "/intermediate_models/dataflow_parent.onnx")
model = ModelWrapper(dataflow_model_filename)

model.save("model.onnx")
# showInNetron("model.onnx")

### 5. Step_Target_fps_parallelization
If target_fps was specified, use the SetFolding transformation to determine parallelization attributes. The auto-generated config will be saved under auto_folding_config.json under the outputs, which can serve as a basis for customizing the folding factors further.

In [20]:
from finn.transformation.fpgadataflow.set_folding import SetFolding
from qonnx.util.config import extract_model_config_to_json

model = ModelWrapper("model.onnx")
target_cycles_per_frame = cfg._resolve_cycles_per_frame()
if target_cycles_per_frame is not None:
    model = model.transform(
        SetFolding(
            target_cycles_per_frame,
            mvau_wwidth_max=cfg.mvau_wwidth_max,
            two_pass_relaxation=cfg.folding_two_pass_relaxation,
        )
    )
    # extract the suggested configuration and save it as json
    hw_attrs = [
        "PE",
        "SIMD",
        "ram_style",
        "resType",
        "mem_mode",
        "runtime_writeable_weights",
    ]
    extract_model_config_to_json(
        model, cfg.output_dir + "/auto_folding_config.json", hw_attrs
    )
    
model.save("model.onnx")

### 6. Step_Apply_folding_config
Apply the folding configuration file onto the model to set folding (parallelization) and other attributes, if config file is specified.

In [21]:
model = ModelWrapper("model.onnx")
if cfg.folding_config_file is not None:
    model = model.transform(GiveUniqueNodeNames())
    model = model.transform(ApplyConfig(cfg.folding_config_file))

if VerificationStepType.FOLDED_HLS_CPPSIM in cfg._resolve_verification_steps():
    # prepare cppsim
    model = model.transform(PrepareCppSim())
    model = model.transform(CompileCppSim())
    model = model.transform(SetExecMode("cppsim"))
    verify_step(model, cfg, "folded_hls_cppsim", need_parent=True)
    
model.save("model.onnx")

### 7. Step_Generate_estimate_report
Generate per-layer resource and cycle estimates using analytical models.

In [22]:
from finn.analysis.fpgadataflow.exp_cycles_per_layer import exp_cycles_per_layer
from finn.analysis.fpgadataflow.op_and_param_counts import (
    aggregate_dict_keys,
    op_and_param_counts,
)

model = ModelWrapper("model.onnx")

if DataflowOutputType.ESTIMATE_REPORTS in cfg.generate_outputs:
    report_dir = cfg.output_dir + "/report"
    os.makedirs(report_dir, exist_ok=True)
    ops_and_params = model.analysis(op_and_param_counts)
    with open(report_dir + "/op_and_param_counts.json", "w") as f:
        json.dump(ops_and_params, f, indent=2)
    estimate_layer_cycles = model.analysis(exp_cycles_per_layer)
    with open(report_dir + "/estimate_layer_cycles.json", "w") as f:
        json.dump(estimate_layer_cycles, f, indent=2)
    estimate_layer_resources = model.analysis(res_estimation)
    estimate_layer_resources["total"] = aggregate_dict_keys(
        estimate_layer_resources
    )
    with open(report_dir + "/estimate_layer_resources.json", "w") as f:
        json.dump(estimate_layer_resources, f, indent=2)
    estimate_layer_resources_complete = model.analysis(res_estimation_complete)
    with open(report_dir + "/estimate_layer_config_alternatives.json", "w") as f:
        json.dump(estimate_layer_resources_complete, f, indent=2)
    # need to call AnnotateCycles before dataflow_performance
    model = model.transform(AnnotateCycles())
    estimate_network_performance = model.analysis(dataflow_performance)
    # add some more metrics to estimated performance
    n_clock_cycles_per_sec = (10**9) / cfg.synth_clk_period_ns
    est_fps = n_clock_cycles_per_sec / estimate_network_performance["max_cycles"]
    estimate_network_performance["estimated_throughput_fps"] = est_fps
    est_latency_ns = (
        estimate_network_performance["critical_path_cycles"]
        * cfg.synth_clk_period_ns
    )
    estimate_network_performance["estimated_latency_ns"] = est_latency_ns
    with open(report_dir + "/estimate_network_performance.json", "w") as f:
        json.dump(estimate_network_performance, f, indent=2)
        
model.save("model.onnx")

/home/miguel_tfg/FINN-v0.9-Stable/src/finn/custom_op/fpgadataflow/streamingmaxpool_batch.py:142: UserWarning: Estimated latency for layer StreamingMaxPool_Batch_0 can be lower than
             actual latency!
  warnings.warn(
/home/miguel_tfg/FINN-v0.9-Stable/src/finn/custom_op/fpgadataflow/streamingmaxpool_batch.py:142: UserWarning: Estimated latency for layer StreamingMaxPool_Batch_1 can be lower than
             actual latency!
  warnings.warn(
/home/miguel_tfg/FINN-v0.9-Stable/src/finn/custom_op/fpgadataflow/streamingmaxpool_batch.py:142: UserWarning: Estimated latency for layer StreamingMaxPool_Batch_2 can be lower than
             actual latency!
  warnings.warn(


In [23]:
! ls {final_output_dir}

intermediate_models  report


In [24]:
! ls {final_output_dir}/report

estimate_layer_config_alternatives.json  estimate_network_performance.json
estimate_layer_cycles.json		 op_and_param_counts.json
estimate_layer_resources.json


In [25]:
! cat {final_output_dir}/report/estimate_network_performance.json

{
  "critical_path_cycles": 10102396,
  "max_cycles": 4718592,
  "max_cycles_node_name": "MatrixVectorActivation_1",
  "estimated_throughput_fps": 21.192762586805557,
  "estimated_latency_ns": 101023960.0
}

In [26]:
import json
def read_json_dict(filename):
    with open(filename, "r") as f:
        ret = json.load(f)
    return ret

In [27]:
read_json_dict(final_output_dir + "/report/estimate_layer_cycles.json")

{'Thresholding_Batch_0': 4096,
 'FMPadding_Batch_0': 4356,
 'ConvolutionInputGenerator_0': 37062,
 'MatrixVectorActivation_0': 589824,
 'StreamingMaxPool_Batch_0': 5120,
 'FMPadding_Batch_1': 1156,
 'ConvolutionInputGenerator_1': 9318,
 'MatrixVectorActivation_1': 4718592,
 'StreamingMaxPool_Batch_1': 1280,
 'FMPadding_Batch_2': 324,
 'ConvolutionInputGenerator_2': 2358,
 'MatrixVectorActivation_2': 4718592,
 'StreamingMaxPool_Batch_2': 320,
 'MatrixVectorActivation_3': 8192,
 'MatrixVectorActivation_4': 1792,
 'LabelSelect_Batch_0': 14}

In [28]:
read_json_dict(final_output_dir + "/report/estimate_layer_resources.json")

{'Thresholding_Batch_0': {'BRAM_18K': 0,
  'BRAM_efficiency': 1,
  'LUT': 8,
  'URAM': 0,
  'URAM_efficiency': 1,
  'DSP': 0},
 'FMPadding_Batch_0': {'BRAM_18K': 0,
  'BRAM_efficiency': 1,
  'LUT': 0,
  'URAM': 0,
  'URAM_efficiency': 1,
  'DSP': 0},
 'ConvolutionInputGenerator_0': {'BRAM_18K': 0,
  'BRAM_efficiency': 1,
  'LUT': 364,
  'URAM': 0,
  'URAM_efficiency': 1,
  'DSP': 0},
 'MatrixVectorActivation_0': {'BRAM_18K': 0,
  'BRAM_efficiency': 1,
  'LUT': 870,
  'URAM': 0,
  'URAM_efficiency': 1,
  'DSP': 1},
 'StreamingMaxPool_Batch_0': {'BRAM_18K': 0,
  'BRAM_efficiency': 1,
  'LUT': 0,
  'URAM': 0,
  'URAM_efficiency': 1,
  'DSP': 0},
 'FMPadding_Batch_1': {'BRAM_18K': 0,
  'BRAM_efficiency': 1,
  'LUT': 0,
  'URAM': 0,
  'URAM_efficiency': 1,
  'DSP': 0},
 'ConvolutionInputGenerator_1': {'BRAM_18K': 0,
  'BRAM_efficiency': 1,
  'LUT': 556,
  'URAM': 0,
  'URAM_efficiency': 1,
  'DSP': 0},
 'MatrixVectorActivation_1': {'BRAM_18K': 0,
  'BRAM_efficiency': 1,
  'LUT': 1142,
  'UR

In [31]:
import os

print("Carpeta actual:", os.getcwd())

os.environ["FINN_BUILD_DIR"] = os.path.abspath("FINN_build")
os.makedirs(os.environ["FINN_BUILD_DIR"], exist_ok=True)

print("FINN_BUILD_DIR =", os.environ["FINN_BUILD_DIR"])

Carpeta actual: /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example
FINN_BUILD_DIR = /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/FINN_build


### 8. Step_HLS_codegen
Generate Vivado HLS code to prepare HLSCustomOp nodes for IP generation.

In [32]:
model = ModelWrapper("model.onnx")
model = model.transform(
    PrepareIP(cfg._resolve_fpga_part(), cfg._resolve_hls_clk_period())
)
model.save("model.onnx")

### 9. Step_HLS_ipgen
Run Vivado HLS synthesis on generated code for HLSCustomOp nodes, in order to generate IP blocks.

In [33]:
model = ModelWrapper("model.onnx")
model = model.transform(HLSSynthIP())
model = model.transform(ReplaceVerilogRelPaths())
report_dir = cfg.output_dir + "/report"
os.makedirs(report_dir, exist_ok=True)
estimate_layer_resources_hls = model.analysis(hls_synth_res_estimation)
with open(report_dir + "/estimate_layer_resources_hls.json", "w") as f:
    json.dump(estimate_layer_resources_hls, f, indent=2)
model.save("model.onnx")

### 10. Step_Set_FIFO_depths
Depending on the auto_fifo_depths setting, do one of the following:
    * if auto_fifo_depths=True:  Run the appropriate auto-sizing transformation to attempt to determine the FIFO sizes that provide full throughput. May take a long time.
    * if auto_fifo_depths=False:  Assume the folding config file contains FIFO sizes as well. Runs the `InsertFIFO` transformation, then `ApplyConfig(cfg.folding_config_file)`, and finally `RemoveShallowFIFOs`. Coherency with config file node naming is ensured by calling `GiveUniqueNodeNames`.

In [34]:
if cfg.auto_fifo_depths:
    if cfg.auto_fifo_strategy == "characterize":
        model = model.transform(InsertDWC())
        model = model.transform(GiveUniqueNodeNames())
        model = model.transform(
            PrepareIP(cfg._resolve_fpga_part(), cfg._resolve_hls_clk_period())
        )
        model = model.transform(HLSSynthIP())
        model = model.transform(PrepareRTLSim())
        model = model.transform(AnnotateCycles())
        period = model.analysis(dataflow_performance)["max_cycles"] + 10
        model = model.transform(DeriveCharacteristic(period))
        model = model.transform(DeriveFIFOSizes())
        model = model.transform(
            InsertFIFO(
                vivado_ram_style=cfg.large_fifo_mem_style,
                max_qsrl_depth=256,
                create_shallow_fifos=True,
            )
        )
        model = model.transform(GiveUniqueNodeNames())
        model = model.transform(GiveReadableTensorNames())
    elif cfg.auto_fifo_strategy == "largefifo_rtlsim":
        # multi-in/out streams currently not supported in our C++ verilator driver
        model_multi_io = len(model.graph.input) > 1 or len(model.graph.output) > 1
        force_python_sim = model_multi_io or cfg.force_python_rtlsim
        if model_multi_io:
            warnings.warn(
                "Multi-in/out streams currently not supported "
                + "in FINN C++ verilator driver, falling back to Python"
            )
        model = model.transform(
            InsertAndSetFIFODepths(
                cfg._resolve_fpga_part(),
                cfg._resolve_hls_clk_period(),
                vivado_ram_style=cfg.large_fifo_mem_style,
                force_python_sim=force_python_sim,
            )
        )
        # InsertAndSetFIFODepths internally removes any shallow FIFOs
        # so no need to call RemoveShallowFIFOs here
    else:
        assert "Unsupported auto_fifo_strategy: " + cfg.auto_fifo_strategy
else:
    # assume folding cfg json contains FIFO sizes too
    # insert DWCs, FIFOs and run ApplyConfig once more
    model = model.transform(InsertDWC())
    # need to make sure all FIFOs are created so that their depth can be
    # set by ApplyConfig, so create_shallow_fifos=True
    model = model.transform(InsertFIFO(create_shallow_fifos=True))
    model = model.transform(GiveUniqueNodeNames())
    model = model.transform(GiveReadableTensorNames())
    if cfg.folding_config_file is not None:
        model = model.transform(ApplyConfig(cfg.folding_config_file))

    # extract the final configuration and save it as json
hw_attrs = [
    "PE",
    "SIMD",
    "ram_style",
    "depth",
    "impl_style",
    "resType",
    "mem_mode",
    "runtime_writeable_weights",
    "inFIFODepths",
    "outFIFODepths",
]
extract_model_config_to_json(
    model, cfg.output_dir + "/final_hw_config.json", hw_attrs
)

# perform FIFO splitting and shallow FIFO removal only after the final config
# json file has been written. otherwise, since these transforms may add/remove
# FIFOs, we get name mismatch problems when trying to reuse the final config.
if cfg.split_large_fifos:
    model = model.transform(SplitLargeFIFOs())
model = model.transform(RemoveShallowFIFOs())

# after FIFOs are ready to go, call PrepareIP and HLSSynthIP again
# this will only run for the new nodes (e.g. FIFOs and DWCs)
model = model.transform(
    PrepareIP(cfg._resolve_fpga_part(), cfg._resolve_hls_clk_period())
)
model = model.transform(HLSSynthIP())
model.save("model.onnx")

/home/miguel_tfg/FINN-v0.9-Stable/src/finn/transformation/fpgadataflow/prepare_ip.py:57: UserWarning: Using pre-existing code for Thresholding_Batch_0
  warnings.warn("Using pre-existing code for %s" % node.name)
/home/miguel_tfg/FINN-v0.9-Stable/src/finn/transformation/fpgadataflow/prepare_ip.py:57: UserWarning: Using pre-existing code for FMPadding_Batch_0
  warnings.warn("Using pre-existing code for %s" % node.name)
/home/miguel_tfg/FINN-v0.9-Stable/src/finn/transformation/fpgadataflow/prepare_ip.py:57: UserWarning: Using pre-existing code for ConvolutionInputGenerator_0
  warnings.warn("Using pre-existing code for %s" % node.name)
/home/miguel_tfg/FINN-v0.9-Stable/src/finn/transformation/fpgadataflow/prepare_ip.py:57: UserWarning: Using pre-existing code for MatrixVectorActivation_0
  warnings.warn("Using pre-existing code for %s" % node.name)
/home/miguel_tfg/FINN-v0.9-Stable/src/finn/transformation/fpgadataflow/prepare_ip.py:57: UserWarning: Using pre-existing code for StreamingM

KeyError: 'VIVADO_PATH'

### . Step_Insert_tlastmarker
If dynamic=True below, will need to specify number of images after which TLAST will be generated by writing to an AXI lite CSR if dynamic=False, always generate TLAST after each image. This must be done after the FIFO depths have been set. 

In [ ]:
model = ModelWrapper("model.onnx")
model = model.transform(InsertTLastMarker(dynamic=False))
model = model.transform(GiveUniqueNodeNames())
model = model.transform(PrepareIP(cfg._resolve_fpga_part(), cfg._resolve_hls_clk_period()))
model = model.transform(HLSSynthIP())
model = model.transform(ReplaceVerilogRelPaths())
model.save("model.onnx")

### 11. Step_Create_stitched_ip
Create stitched IP for a graph after all HLS IP blocks have been generated. Depends on the DataflowOutputType.STITCHED_IP output product.

In [ ]:
model = ModelWrapper("model.onnx")
if DataflowOutputType.STITCHED_IP in cfg.generate_outputs:
    stitched_ip_dir = cfg.output_dir + "/stitched_ip"
    model = model.transform(CreateStitchedIP(cfg._resolve_fpga_part(),cfg.synth_clk_period_ns,vitis=cfg.stitched_ip_gen_dcp,signature=cfg.signature,))
    # TODO copy all ip sources into output dir? as zip?
    copy_tree(model.get_metadata_prop("vivado_stitch_proj"), stitched_ip_dir)
    print("Vivado stitched IP written into " + stitched_ip_dir)
if VerificationStepType.STITCHED_IP_RTLSIM in cfg._resolve_verification_steps():
    # prepare ip-stitched rtlsim
    verify_model = deepcopy(model)
    verify_model = prepare_for_stitched_ip_rtlsim(verify_model, cfg)
    # use critical path estimate to set rtlsim liveness threshold
    # (very conservative)
    verify_model = verify_model.transform(AnnotateCycles())
    estimate_network_performance = verify_model.analysis(dataflow_performance)
    prev_liveness = pyverilate_get_liveness_threshold_cycles()
    os.environ["LIVENESS_THRESHOLD"] = str(
        int(estimate_network_performance["critical_path_cycles"])
    )
    if cfg.verify_save_rtlsim_waveforms:
        report_dir = cfg.output_dir + "/report"
        os.makedirs(report_dir, exist_ok=True)
        verify_model.set_metadata_prop(
            "rtlsim_trace", "%s/verify_rtlsim.vcd" % (report_dir)
        )
    verify_step(verify_model, cfg, "stitched_ip_rtlsim", need_parent=True)
    os.environ["LIVENESS_THRESHOLD"] = str(prev_liveness)

model.save("model.onnx")

### 12. Step_Measure_RTLsim_performance
Measure performance + latency of stitched-IP model in rtlsim (pyverilator). Depends on the DataflowOutputType.STITCHED_IP output product.

In [ ]:
model = ModelWrapper("model.onnx")
if DataflowOutputType.RTLSIM_PERFORMANCE in cfg.generate_outputs:
    assert (
        DataflowOutputType.STITCHED_IP in cfg.generate_outputs
    ), "rtlsim_perf needs stitched IP"
    report_dir = cfg.output_dir + "/report"
    os.makedirs(report_dir, exist_ok=True)
    # prepare ip-stitched rtlsim
    rtlsim_model = deepcopy(model)
    rtlsim_model = prepare_for_stitched_ip_rtlsim(rtlsim_model, cfg)
    # run with single input to get latency
    orig_rtlsim_trace_depth = get_rtlsim_trace_depth()
    rtlsim_bs = int(cfg.rtlsim_batch_size)
    assert rtlsim_bs > 0, "rtlsim batch size must be >0"
    if cfg.verify_save_rtlsim_waveforms:
        # set depth to 3 for layer-by-layer visibility
        os.environ["RTLSIM_TRACE_DEPTH"] = "3"
        rtlsim_model.set_metadata_prop(
            "rtlsim_trace", "%s/rtlsim_perf_batch_%d.vcd" % (report_dir, rtlsim_bs)
        )
    rtlsim_model.set_metadata_prop("extra_verilator_args", str(["-CFLAGS", "-O3"]))
    rtlsim_perf_dict = throughput_test_rtlsim(rtlsim_model, rtlsim_bs)
    rtlsim_latency = rtlsim_perf_dict["cycles"]
    rtlsim_perf_dict["latency_cycles"] = rtlsim_latency
    with open(report_dir + "/rtlsim_performance.json", "w") as f:
        json.dump(rtlsim_perf_dict, f, indent=2)
    if cfg.verify_save_rtlsim_waveforms:
        # restore original trace depth
        os.environ["RTLSIM_TRACE_DEPTH"] = str(orig_rtlsim_trace_depth)
        
model.save("model.onnx")

### 14. Step_OOC_synthesis
Run out-of-context synthesis and generate reports. Depends on the DataflowOutputType.STITCHED_IP output product.

In [ ]:
model = ModelWrapper("model.onnx")
if DataflowOutputType.OOC_SYNTH in cfg.generate_outputs:
    assert (
        DataflowOutputType.STITCHED_IP in cfg.generate_outputs
    ), "OOC needs stitched IP"
    model = model.transform(
        SynthOutOfContext(
            part=cfg._resolve_fpga_part(), clk_period_ns=cfg.synth_clk_period_ns
        )
    )
    report_dir = cfg.output_dir + "/report"
    os.makedirs(report_dir, exist_ok=True)
    ooc_res_dict = model.get_metadata_prop("res_total_ooc_synth")
    ooc_res_dict = eval(ooc_res_dict)

    estimate_network_performance = model.analysis(dataflow_performance)
    # add some more metrics to estimated performance
    n_clock_cycles_per_sec = float(ooc_res_dict["fmax_mhz"]) * (10**6)
    est_fps = n_clock_cycles_per_sec / estimate_network_performance["max_cycles"]
    ooc_res_dict["estimated_throughput_fps"] = est_fps
    with open(report_dir + "/ooc_synth_and_timing.json", "w") as f:
        json.dump(ooc_res_dict, f, indent=2)
        
model.save("model.onnx")

In [ ]:
! ls {final_output_dir}/stitched_ip

In [ ]:
! ls {final_output_dir}/report

In [ ]:
! cat {final_output_dir}/report/ooc_synth_and_timing.json

In [ ]:
! cat {final_output_dir}/report/rtlsim_performance.json

In [ ]:
! cat {final_output_dir}/final_hw_config.json

### 15. Step_Synthesize_bitfile
Synthesize a bitfile for the using the specified shell flow, using either Vivado or Vitis, to target the specified board.

In [ ]:
# model = ModelWrapper("model.onnx")
# if DataflowOutputType.BITFILE in cfg.generate_outputs:
#     bitfile_dir = cfg.output_dir + "/bitfile"
#     os.makedirs(bitfile_dir, exist_ok=True)
#     report_dir = cfg.output_dir + "/report"
#     os.makedirs(report_dir, exist_ok=True)
#     partition_model_dir = cfg.output_dir + "/intermediate_models/kernel_partitions"
#     if cfg.shell_flow_type == ShellFlowType.VIVADO_ZYNQ:
#         model = model.transform(
#             ZynqBuild(
#                 cfg.board,
#                 cfg.synth_clk_period_ns,
#                 cfg.enable_hw_debug,
#                 partition_model_dir=partition_model_dir,
#             )
#         )
#         copy(model.get_metadata_prop("bitfile"), bitfile_dir + "/finn-accel.bit")
#         copy(model.get_metadata_prop("hw_handoff"), bitfile_dir + "/finn-accel.hwh")
#         copy(
#             model.get_metadata_prop("vivado_synth_rpt"),
#             report_dir + "/post_synth_resources.xml",
#         )
#         vivado_pynq_proj_dir = model.get_metadata_prop("vivado_pynq_proj")
#         timing_rpt = (
#             "%s/finn_zynq_link.runs/impl_1/top_wrapper_timing_summary_routed.rpt"
#             % vivado_pynq_proj_dir
#         )
#         copy(timing_rpt, report_dir + "/post_route_timing.rpt")

#     elif cfg.shell_flow_type == ShellFlowType.VITIS_ALVEO:
#         model = model.transform(
#             VitisBuild(
#                 cfg._resolve_fpga_part(),
#                 cfg.synth_clk_period_ns,
#                 cfg._resolve_vitis_platform(),
#                 strategy=cfg._resolve_vitis_opt_strategy(),
#                 enable_debug=cfg.enable_hw_debug,
#                 floorplan_file=cfg.vitis_floorplan_file,
#                 partition_model_dir=partition_model_dir,
#             )
#         )
#         copy(model.get_metadata_prop("bitfile"), bitfile_dir + "/finn-accel.xclbin")
#         copy(
#             model.get_metadata_prop("vivado_synth_rpt"),
#             report_dir + "/post_synth_resources.xml",
#         )
#     else:
#         raise Exception("Unrecognized shell_flow_type: " + str(cfg.shell_flow_type))
#     print("Bitfile written into " + bitfile_dir)
    
# model.save("model.onnx")

### 13. Step_Make_pynq_driver
Create a PYNQ Python driver that can be used to interface the generated accelerator.

In [ ]:
# model = ModelWrapper("model.onnx")
# if DataflowOutputType.PYNQ_DRIVER in cfg.generate_outputs:
#     driver_dir = cfg.output_dir + "/driver"
#     model = model.transform(MakePYNQDriver(cfg._resolve_driver_platform()))
#     copy_tree(model.get_metadata_prop("pynq_driver_dir"), driver_dir)
#     print("PYNQ Python driver written into " + driver_dir)
    
# model.save("model.onnx")

### 16. Step_Deployment_package
Create a deployment package including the driver and bitfile.

In [ ]:
# model = ModelWrapper("model.onnx")
# if DataflowOutputType.DEPLOYMENT_PACKAGE in cfg.generate_outputs:
#     deploy_dir = cfg.output_dir + "/deploy"
#     bitfile_dir = cfg.output_dir + "/bitfile"
#     driver_dir = cfg.output_dir + "/driver"
#     os.makedirs(deploy_dir, exist_ok=True)
#     copy_tree(bitfile_dir, deploy_dir + "/bitfile")
#     copy_tree(driver_dir, deploy_dir + "/driver")
    
# model.save("model.onnx")